# Phase 1: Quasi-Experimental Promotional Analysis

In this notebook, we analyze the promotional impact on sales for the Dominick's Finer Foods dataset. We will merge the product descriptions (`upccer.csv`) with the weekly sales data (`wcer.csv`).

> **Note:** We approach this analysis with strict statistical humility. We are estimating partial effects controlling for observed confounders rather than definitively proving causal impact.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from linearmodels.iv import AbsorbingLS
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Ensure processed data directory exists
os.makedirs('../data/processed', exist_ok=True)

## 1. Data Loading and Cleaning
We load the tables and merge them on `UPC`. To ensure memory efficiency, we optimize data types during or immediately after loading.

In [ ]:
# Load data with memory-efficient dtype assumptions where possible
upc_path = '../data/raw/upccer.csv'
wcer_path = '../data/raw/wcer.csv'

dtypes_wcer = {
    'STORE': 'int32',
    'WEEK': 'int16',
    'MOVE': 'int32',
    'QTY': 'int16',
    'PRICE': 'float32',
    'SALE': 'category',
    'PROFIT': 'float32',
    'OK': 'int8'
}

print("Loading Data...")
# Using encoding='latin1' to handle special characters common in legacy Dominick's datasets
df_upc = pd.read_csv(upc_path, dtype=str, encoding='latin1')
df_wcer = pd.read_csv(wcer_path, dtype=dtypes_wcer, low_memory=False, encoding='latin1')

print("Cleaning UPC keys for merging...")
# Normalize UPCs: cast to string and strip whitespace to ensure match
df_upc['UPC'] = df_upc['UPC'].astype(str).str.strip()
df_wcer['UPC'] = df_wcer['UPC'].astype(str).str.strip()

# Merge datasets on UPC
df = df_wcer.merge(df_upc, on='UPC', how='inner')

print(f"Merged data shape: {df.shape}")

## 2. Feature Engineering & Preprocessing
We prepare our variables for the regression model, treating the data as observational. We will control for price variation, baseline demand (Store FE), and seasonality (Month/Week FE).

In [ ]:
# Rename MOVE to Sales if it's the target
if 'MOVE' in df.columns:
    df['Sales'] = df['MOVE']

# Create an explicit Promo indicator
df['Promo'] = df['SALE'].notna() & (df['SALE'].astype(str) != ' ') & (df['SALE'].astype(str) != 'nan')
df['Promo'] = df['Promo'].astype('int8')

# Derive monthproxy from WEEK
df['Month_Approx'] = (df['WEEK'] / 4).astype(int)

# Categoricals for Fixed Effects
df['Store_Cat'] = df['STORE'].astype('category')
df['Month_Cat'] = df['Month_Approx'].astype('category')

## 3. Deep Exploratory Data Analysis (EDA)
Before modeling, we must understand the intricacies of our dataset. We evaluate core dimensions like price elasticity and temporal dynamics.

In [ ]:
sns.set_theme(style="whitegrid")

# Create a subset for visualization
df_viz = df[(df['Sales'] > 0) & (df['Sales'] < df['Sales'].quantile(0.99))].copy()
df_viz['Promo_Label'] = df_viz['Promo'].map({0: 'No Promo', 1: 'On Promo'})

fig = plt.figure(figsize=(20, 15))
ax1 = plt.subplot(2, 2, 1)
sns.kdeplot(data=df_viz, x='Sales', hue='Promo_Label', fill=True, alpha=0.5, ax=ax1)
ax1.set_title('1. Sales Density: Promoted vs Non-Promoted', fontweight='bold')

df_viz['Log_Sales'] = np.log(df_viz['Sales'])
df_viz['Log_Price'] = np.log(df_viz['PRICE'])
ax2 = plt.subplot(2, 2, 2)
sns.scatterplot(data=df_viz.sample(n=min(10000, len(df_viz)), random_state=42), 
                x='Log_Price', y='Log_Sales', hue='Promo_Label', alpha=0.4, ax=ax2)
ax2.set_title('2. Price Elasticity: Log(Sales) vs Log(Price)', fontweight='bold')

ax3 = plt.subplot(2, 2, 3)
weekly_trend = df_viz.groupby(['WEEK', 'Promo_Label'])['Sales'].mean().reset_index()
sns.lineplot(data=weekly_trend, x='WEEK', y='Sales', hue='Promo_Label', ax=ax3)
ax3.set_title('3. Average Weekly Sales Over Time', fontweight='bold')

ax4 = plt.subplot(2, 2, 4)
store_counts = df_viz.groupby('STORE')['Promo'].mean().sort_values(ascending=False).head(20).reset_index()
sns.barplot(data=store_counts, x='STORE', y='Promo', order=store_counts['STORE'], palette='magma', ax=ax4)
ax4.set_title('4. Structural Variance: Top 20 Stores by Promo Frequency', fontweight='bold')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Quasi-Experimental Promotional Analysis (Memory-Efficient)
We use **Industry Standard Fixed Effect Absorption** to handle the large dataset (~5M rows). High-dimensional fixed effects (Store and Month) are "absorbed" rather than creating thousands of dummy variables.

In [ ]:
from linearmodels.iv import AbsorbingLS

# 1. Prepare Data
df_model = df[df['Sales'] > 0].copy()
df_model['Log_Sales'] = np.log(df_model['Sales'])
df_model['Log_Price'] = np.log(df_model['PRICE'])
df_model['Promo_Price_Interaction'] = df_model['Promo'] * df_model['Log_Price']
# Add constant for the intercept
df_model['const'] = 1.0

dependent = df_model['Log_Sales']
exog = df_model[['const', 'Promo', 'Log_Price', 'Promo_Price_Interaction']]

# 2. Specify Absorbed Fixed Effects (Store and Month)
# This is mathematically equivalent to OLS with dummy variables but uses a fraction of the memory.
absorb = df_model[['Store_Cat', 'Month_Cat']]

print("Fitting Industry-Standard Fixed Effects Model (Absorbing LS)...")
model = AbsorbingLS(dependent, exog, absorb=absorb)
res = model.fit()

print("========== Regression Results ==========")
print(res.summary)

## 5. Exporting Processed Data
To ensure Phase 2 has access to the cleaned and merged variables, we export the final dataset.

In [ ]:
processed_path = '../data/processed/cleaned_data.csv'
print(f"Exporting cleaned data to {processed_path}...")
df.to_csv(processed_path, index=False)
print("Export Complete.")